# Task 2
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, DotProduct, RBF, RationalQuadratic, Matern
from sklearn.model_selection import KFold, cross_val_score

# Data Loading
TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test
(and potentially change initialization of variables to accomodate how you deal with non-numeric data)

In [ ]:
"""
This loads the training and test data, preprocesses it, removes the NaN
values and interpolates the missing data using imputation

Parameters
----------
Compute
----------
X_train: matrix of floats, training input with features
y_train: array of floats, training output with labels
X_test: matrix of floats: dim = (100, ?), test input with features
"""
# Load training data
train_df = pd.read_csv("train.csv")

print("Training data:")
print("Shape:", train_df.shape)
print(train_df.head(2))
print('\n')

# Load test data
test_df = pd.read_csv("test.csv")

print("Test data:")
print(test_df.shape)
print(test_df.head(2))

# Keep all rows with missing feature values and remove only rows with missing target
train_df = train_df.dropna(subset=["price_CHF"]).reset_index(drop=True)

# Extract raw features and target
X_train_raw = train_df.drop(columns=["price_CHF"])
y_train = train_df["price_CHF"].to_numpy()
X_test_raw = test_df.copy()

# Identify column types
categorical_features = ["season"]
numeric_features = [col for col in X_train_raw.columns if col not in categorical_features]

# Preprocessing:
# - numeric columns: KNN imputation + scaling
# - categorical columns: most frequent imputation + one-hot encoding
numeric_transformer = Pipeline(steps=[
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Fit preprocessing on train and transform both train and test
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)


assert (X_train.shape[1] == X_test.shape[1]) and (X_train.shape[0] == y_train.shape[0]) and (X_test.shape[0] == 100), "Invalid data shape"

print("\nPreprocessed training shape:", X_train.shape)
print("Preprocessed test shape:", X_test.shape)

# Modeling and Prediction
TODO: Define the model and fit it using training data. Then, use test data to make predictions

In [ ]:
"""
This defines the model, fits training data and then does the prediction
with the test data 

Parameters
----------
X_train: matrix of floats, training input with 10 features
y_train: array of floats, training output
X_test: matrix of floats: dim = (100, ?), test input with 10 features

Compute
----------
y_test: array of floats: dim = (100,), predictions on test set
"""
class Model(object):
    def __init__(self):
        super().__init__()
        self.model = None
        self.best_kernel_name = None
        self.best_score = -np.inf

    def fit(self, X_train: np.ndarray, y_train: np.ndarray):
        kernels = {
            "DotProduct + RBF":
                ConstantKernel(1.0, (1e-3, 1e3)) * (DotProduct() + RBF()),
            "DotProduct + RationalQuadratic":
                ConstantKernel(1.0, (1e-3, 1e3)) * (DotProduct() + RationalQuadratic()),
            "DotProduct + Matern":
                ConstantKernel(1.0, (1e-3, 1e3)) * (DotProduct() + Matern(nu=1.5)),
        }

        cv = KFold(n_splits=5, shuffle=True, random_state=42)

        for kernel_name, kernel in kernels.items():
            gpr = GaussianProcessRegressor(
                kernel=kernel,
                random_state=42,
                n_restarts_optimizer=3
            )

            scores = cross_val_score(
                gpr,
                X_train,
                y_train,
                cv=cv,
                scoring="r2",
                n_jobs=None
            )

            print(kernel_name)
            print("Fold R^2 scores:", scores)
            print("Mean R^2:", scores.mean())
            print("Std R^2:", scores.std())
            print("-" * 50)

            if scores.mean() > self.best_score:
                self.best_score = scores.mean()
                self.best_kernel_name = kernel_name
                self.model = GaussianProcessRegressor(
                    kernel=kernel,
                    random_state=42,
                    n_restarts_optimizer=3
                )

        print("Best kernel:", self.best_kernel_name)
        print("Best CV Mean R^2:", self.best_score)

        self.model.fit(X_train, y_train)

    def predict(self, X_test: np.ndarray) -> np.ndarray:
        y_pred = self.model.predict(X_test)
        assert y_pred.shape == (X_test.shape[0],), "Invalid data shape"
        return y_pred

In [ ]:
model = Model()
# Use this function to fit the model
model.fit(X_train=X_train, y_train=y_train)
# Use this function for inference
y_pred = model.predict(X_test)

# Saving Results
You don't have to change this

In [ ]:
dt = pd.DataFrame(y_pred) 
dt.columns = ['price_CHF']
dt.to_csv('results.csv', index=False)
print("\nResults file successfully generated!")